# Hyperparameter Tunning

In [4]:
# EDA

import os
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader , TensorDataset
import optuna

sys.path.append(os.path.abspath("../src"))
sys.path.append(os.path.abspath(".."))

from preprocessing import get_X_y

C:\Users\sailj\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Data Load

df_train = pd.read_csv("../dataset/fashion-mnist_train.csv")

X_train , y_train = get_X_y(df_train)

df_test = pd.read_csv("../dataset/fashion-mnist_test.csv")

X_test , y_test = get_X_y(df_test)

X_train

,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,5,0,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,1,2,0,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59996,0,0,0,0,0,0,0,0,0,0,...,73,0,0,0,0,0,0,0,0,0
59997,0,0,0,0,0,0,0,0,0,0,...,160,162,163,135,94,0,0,0,0,0
59998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
# Data Prep

X_test , X_val , y_test , y_val = train_test_split(
    X_test ,
    y_test ,
    random_state = 42 ,
    stratify = y_test ,
    test_size = 0.5
)

X_train = X_train / 255.0
X_test = X_test / 255.0
X_val = X_val / 225.0

X_train = torch.tensor(
    X_train.values ,
    dtype = torch.float32
)
y_train = torch.tensor(
    y_train.values ,
    dtype = torch.long
)
X_val = torch.tensor(
    X_val.values ,
    dtype = torch.float32
)
y_val = torch.tensor(
    y_val.values ,
    dtype = torch.long
)

train_dataset = TensorDataset(
    X_train ,
    y_train
)
val_dataset = TensorDataset(
    X_val ,
    y_val
)

In [11]:
# Model Builder

def build_model(hidden1 , hidden2 , hidden3 , hidden4 , dropout1 , dropout2 , dropout3 , dropout4) :

    model = nn.Sequential(
    
        nn.Linear(784 , hidden1) ,
        nn.BatchNorm1d(hidden1) ,
        nn.LeakyReLU() ,
        nn.Dropout(dropout1) ,
    
        nn.Linear(hidden1 , hidden2) ,
        nn.BatchNorm1d(hidden2) ,
        nn.LeakyReLU() ,
        nn.Dropout(dropout2) ,
    
        nn.Linear(hidden2 , hidden3) ,
        nn.BatchNorm1d(hidden3) ,
        nn.LeakyReLU() ,
        nn.Dropout(dropout3) ,
    
        nn.Linear(hidden3 , hidden4) ,
        nn.BatchNorm1d(hidden4) ,
        nn.LeakyReLU() ,
        nn.Dropout(dropout4) ,
    
        nn.Linear(hidden4 , 10) 
    )

    return model

def init_weight(m) : 

    if isinstance(m , nn.Linear) :
        nn.init.kaiming_normal(
            m.weight ,
            mode = "fan_in" ,
            nonlinearity = "leaky_relu"
        )
        nn.init.zeros_(m.bias)

In [12]:
# Objective 

def objective(trial) :

    hidden1 = trial.suggest_int(
        "hidden1" ,
        128 ,
        256 ,
        step = 32
    )
    hidden2 = trial.suggest_int(
        "hidden2" ,
        64 ,
        128,
        step = 16
    )
    hidden3 = trial.suggest_int(
        "hidden3" ,
        16 ,
        64 ,
        step = 8
    )
    hidden4 = trial.suggest_int(
        "hidden4" ,
        16 ,
        32 ,
        step = 8
    )
    dropout1 = trial.suggest_float(
        "dropout1" ,
        0.0 ,
        0.5
    )
    dropout2 = trial.suggest_float(
        "dropout2" ,
        0.0 ,
        0.5
    )
    dropout3 = trial.suggest_float(
        "dropout3" ,
        0.0 ,
        0.5
    )
    dropout4 = trial.suggest_float(
        "dropout4" ,
        0.0 ,
        0.5 ,
        step = 0.1
    )
    lr = trial.suggest_float(
        "lr" ,
        1e-4 ,
        1e-2 ,
        log = True
    )
    weight_decay = trial.suggest_float(
        "weight_decay" ,
        1e-5 ,
        1e-2 ,
        log = True
    )
    batch_size = trial.suggest_categorical(
        "batch_size" ,
        [32 , 64 , 128 , 256]
    )

    model = build_model(hidden1 , hidden2 , hidden3 , hidden4 , dropout1 , dropout2 , dropout3 , dropout4)

    model.apply(init_weight)

    optimizer = optim.AdamW(
    model.parameters() ,
    lr = lr ,
    weight_decay = weight_decay
    )

    critrion = nn.CrossEntropyLoss()

    train_loader = DataLoader(
        train_dataset ,
        batch_size = batch_size ,
        shuffle = True
    )
    
    val_loader = DataLoader(
        val_dataset ,
        batch_size = batch_size ,
        shuffle = False
    )

    for epoch in range(100) :

        model.train()

        for X_batch , y_batch in train_loader :

            optimizer.zero_grad()
            
            logits = model(X_batch)

            loss = critrion(
                logits ,
                y_batch
            )

            loss.backward()

            optimizer.step()

        val_loss = 0.0

        with torch.no_grad() :

            for X_batch , y_batch in val_loader :

                logits = model(X_batch)
                
                loss = critrion(
                    logits ,
                    y_batch
                )

                val_loss += loss.item()

            val_loss /= len(val_loader)

            trial.report(
                val_loss ,
                epoch
            )

            if trial.should_prune() :

                raise optuna.TrialPruned()

        return val_loss

In [ ]:
# Study 1

study = optuna.create_study(
    study_name="STUDY 1",
    direction = "minimize"
)

study.optimize(
    objective ,
    n_trials = 50
)

print(study.best_params)
print()
print(study.best_value)

[I 2026-09-05 22:20:41,823] A new study created in memory with name: STUDY 1
C:\Users\sailj\AppData\Local\Temp\ipykernel_17084\2683029419.py:35: FutureWarning: `nn.init.kaiming_normal` is now deprecated in favor of `nn.init.kaiming_normal_`.
  nn.init.kaiming_normal(
[I 2026-09-05 22:20:44,673] Trial 0 finished with value: 0.9994722187519074 and parameters: {'hidden1': 256, 'hidden2': 80, 'hidden3': 16, 'hidden4': 32, 'dropout1': 0.09164971340758826, 'dropout2': 0.023079189093558994, 'dropout3': 0.4791468231904985, 'dropout4': 0.1, 'lr': 0.0007910974955217513, 'weight_decay': 1.691491894035872e-05, 'batch_size': 256}. Best is trial 0 with value: 0.9994722187519074.
[I 2026-09-05 22:20:51,132] Trial 1 finished with value: 0.6251891813700712 and parameters: {'hidden1': 192, 'hidden2': 96, 'hidden3': 32, 'hidden4': 32, 'dropout1': 0.007290632865980695, 'dropout2': 0.03480068656537544, 'dropout3': 0.17300430839255432, 'dropout4': 0.2, 'lr': 0.0003206369158722416, 'weight_decay': 0.00019737

In [13]:
# Study 1

study = optuna.create_study(
    study_name="STUDY 1",
    direction = "minimize"
)

study.optimize(
    objective ,
    n_trials = 50
)

print(study.best_params)
print()
print(study.best_value)

[I 2026-09-05 21:50:31,732] A new study created in memory with name: STUDY 1
C:\Users\sailj\AppData\Local\Temp\ipykernel_17084\2683029419.py:35: FutureWarning: `nn.init.kaiming_normal` is now deprecated in favor of `nn.init.kaiming_normal_`.
  nn.init.kaiming_normal(
[I 2026-09-05 21:50:34,592] Trial 0 finished with value: 1.583198431134224 and parameters: {'hidden1': 160, 'hidden2': 96, 'hidden3': 40, 'hidden4': 16, 'dropout1': 0.01624019088615275, 'dropout2': 0.1709573784203725, 'dropout3': 0.3996743136137644, 'dropout4': 0.4, 'lr': 0.00010575772522444312, 'weight_decay': 4.792864419213004e-05, 'batch_size': 128}. Best is trial 0 with value: 1.583198431134224.
[I 2026-09-05 21:50:36,924] Trial 1 finished with value: 0.5152569606900215 and parameters: {'hidden1': 256, 'hidden2': 128, 'hidden3': 40, 'hidden4': 16, 'dropout1': 0.1512188676802078, 'dropout2': 0.08949753732375165, 'dropout3': 0.30509813505860134, 'dropout4': 0.1, 'lr': 0.006598965467678558, 'weight_decay': 0.0029363062977

{'hidden1': 256, 'hidden2': 128, 'hidden3': 24, 'hidden4': 24, 'dropout1': 0.03651566135425316, 'dropout2': 0.04011799029039411, 'dropout3': 0.03255362371798396, 'dropout4': 0.0, 'lr': 0.004291440303963457, 'weight_decay': 4.415561505155581e-05, 'batch_size': 256}

0.406993967294693


# Fianl Verdict 

# ----------------------------------------------------------------

# 'hidden1': 256 
# 'hidden2': 128
# 'hidden3': 24
# 'hidden4': 24
# 'dropout1': 0.03651566135425316
# 'dropout2': 0.04011799029039411
# 'dropout3': 0.03255362371798396
# 'dropout4': 0.0
# 'lr': 0.004291440303963457
# 'weight_decay': 4.415561505155581e-05
# 'batch_size': 256